# 🌱 BioVision: Deep Learning Model Training (Google Colab GPU)
**Course:** SE-314L Artificial Intelligence  
**Backbone:** EfficientNetB0 Transfer Learning with Two-Phase Fine-Tuning  

### ⚡ Quick Guide:
1. **Enable Free GPU:** Go to **Runtime** > **Change runtime type** > Select **T4 GPU** > Click **Save**.
2. **Dataset:** Run the dataset setup cell below (supports both uploaded zip and automatic Kaggle download).
3. **Train:** Run the training cell (~8–10 minutes).
4. **Download:** The last cell automatically downloads `biovision_trained_artifacts.zip` containing `disease_model.keras`, `class_names.json`, and all evaluation plots.

In [ ]:
# 1. Verify GPU Acceleration
!nvidia-smi
!pip install -q tensorflow scikit-learn seaborn matplotlib pandas pillow kagglehub

### 📁 Step 2: Prepare Dataset (Robust Auto-Detector)

In [ ]:
import os, shutil, zipfile
from pathlib import Path

# Check if dataset_split.zip was uploaded or exists in /content
if not os.path.exists('/content/dataset_split.zip') and not os.path.exists('/content/data/split/train'):
    print("Choose how to load data:")
    print("1) Upload 'dataset_split.zip' from your laptop, OR")
    print("2) Click Cancel/skip to automatically download from Kaggle.")
    try:
        from google.colab import files
        uploaded = files.upload()
    except Exception as e:
        print("Upload skipped.")

# Handle unzipping if dataset_split.zip is present
if os.path.exists('/content/dataset_split.zip') and not os.path.exists('/content/data/split/train'):
    print("Extracting dataset_split.zip directly into /content/data...")
    !unzip -q /content/dataset_split.zip -d /content/data
    
# Fix nested directory if it extracted to /content/data/split/split
if os.path.exists('/content/data/split/split/train'):
    print("Fixing nested directory structure...")
    !mv /content/data/split/split/* /content/data/split/
    !rm -rf /content/data/split/split

# If still not found, download via kagglehub automatically
if not os.path.exists('/content/data/split/train'):
    print("Downloading PlantVillage via KaggleHub (no API token required)...")
    import kagglehub, random
    from sklearn.model_selection import train_test_split
    
    path = kagglehub.dataset_download("abdallahalidev/plantvillage-dataset")
    print(f"Downloaded to: {path}")
    
    raw_color = None
    for p in Path(path).rglob('color'):
        if p.is_dir():
            raw_color = p
            break
            
    OUT = Path('/content/data/split')
    OUT.mkdir(parents=True, exist_ok=True)
    
    INCLUDE = []  # Empty list = Train on all 38 classes across all 14 crops!
    MAX_PER_CLASS = 600
    SEED = 42
    
    for cls_dir in sorted(p for p in raw_color.iterdir() if p.is_dir()):
        if INCLUDE and not any(cls_dir.name.startswith(c) for c in INCLUDE):
            continue
        files_list = sorted(f for f in cls_dir.iterdir() if f.suffix.lower() in ('.jpg', '.jpeg', '.png'))
        if not files_list:
            continue
        random.Random(SEED).shuffle(files_list)
        if MAX_PER_CLASS:
            files_list = files_list[:MAX_PER_CLASS]
        train, rest = train_test_split(files_list, test_size=0.30, random_state=SEED)
        val, test = train_test_split(rest, test_size=0.50, random_state=SEED)
        for split, items in (('train', train), ('val', val), ('test', test)):
            dest = OUT / split / cls_dir.name
            dest.mkdir(parents=True, exist_ok=True)
            for f in items:
                shutil.copy2(f, dest / f.name)
        print(f"{cls_dir.name}: Train={len(train)}, Val={len(val)}, Test={len(test)}")

# Final verification
if os.path.exists('/content/data/split/train'):
    classes = os.listdir('/content/data/split/train')
    print(f"\nSUCCESS! Dataset ready at /content/data/split with {len(classes)} classes:")
    print(classes)
else:
    print("ERROR: Could not establish /content/data/split/train.")

### 🧠 Step 3: Train & Fine-Tune EfficientNetB0 (~8–10 minutes on T4 GPU)

In [ ]:
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import classification_report, confusion_matrix, f1_score

DATA = '/content/data/split'
IMG = (224, 224)
BATCH = 32
EPOCHS_HEAD = 5
EPOCHS_FINE = 8
AUTOTUNE = tf.data.AUTOTUNE

keras.utils.set_random_seed(42)
Path('/content/models').mkdir(exist_ok=True)
Path('/content/reports').mkdir(exist_ok=True)

def load_ds(split, shuffle):
    return keras.utils.image_dataset_from_directory(
        f'{DATA}/{split}',
        image_size=IMG,
        batch_size=BATCH,
        shuffle=shuffle,
        seed=42
    )

train_raw = load_ds('train', True)
class_names = train_raw.class_names
val_ds = load_ds('val', False).prefetch(AUTOTUNE)
test_ds = load_ds('test', False).prefetch(AUTOTUNE)

print(f"Loaded {len(class_names)} classes: {class_names}")

augment = keras.Sequential([
    layers.RandomFlip('horizontal_and_vertical'),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
])

train_ds = train_raw.map(
    lambda x, y: (augment(x, training=True), y),
    num_parallel_calls=AUTOTUNE
).prefetch(AUTOTUNE)

# EfficientNetB0 transfer learning backbone
base = keras.applications.EfficientNetB0(
    include_top=False,
    weights='imagenet',
    input_shape=IMG + (3,)
)
base.trainable = False

x = layers.GlobalAveragePooling2D()(base.output)
x = layers.Dropout(0.3)(x)
out = layers.Dense(len(class_names), activation='softmax')(x)
model = keras.Model(base.input, out)

cbs = [
    keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(factor=0.3, patience=1)
]

# Phase 1: Train classification head
print("\n--- Phase 1: Training Head ---")
model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
h1 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_HEAD, callbacks=cbs)

# Phase 2: Fine-tune top 30 layers
print("\n--- Phase 2: Fine-Tuning Top 30 Layers ---")
for layer in base.layers[-30:]:
    if not isinstance(layer, layers.BatchNormalization):
        layer.trainable = True

model.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
h2 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_FINE, callbacks=cbs)

# Plot Learning Curves
acc = h1.history['accuracy'] + h2.history['accuracy']
val = h1.history['val_accuracy'] + h2.history['val_accuracy']
plt.figure(figsize=(7, 4))
plt.plot(acc, label='Train Accuracy', color='#1f77b4', lw=2)
plt.plot(val, label='Validation Accuracy', color='#ff7f0e', lw=2)
plt.axvline(len(h1.history['accuracy']) - 1, linestyle='--', color='gray', label='Fine-Tuning Start')
plt.xlabel('Epoch'); plt.ylabel('Accuracy'); plt.legend(); plt.tight_layout()
plt.savefig('/content/reports/training_curves.png', dpi=150)
plt.show(); plt.close()

# Evaluate on held-out test set
print("\n--- Evaluating on Held-out Test Set ---")
y_true = np.concatenate([y.numpy() for _, y in test_ds])
y_pred = np.argmax(model.predict(test_ds, verbose=0), axis=1)

rep = classification_report(y_true, y_pred, target_names=class_names, digits=3)
Path('/content/reports/classification_report.txt').write_text(rep)
print(rep)

cm = confusion_matrix(y_true, y_pred)
size = max(7, len(class_names) * 0.55)
plt.figure(figsize=(size, size))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.xticks(rotation=45, ha='right'); plt.tight_layout()
plt.savefig('/content/reports/confusion_matrix.png', dpi=150)
plt.show(); plt.close()

metrics = {
    'test_accuracy': float((y_true == y_pred).mean()),
    'macro_f1': float(f1_score(y_true, y_pred, average='macro')),
    'n_classes': len(class_names),
    'n_test_images': int(len(y_true))
}
Path('/content/reports/classifier_metrics.json').write_text(json.dumps(metrics, indent=2))
print("Final Classifier Metrics:", json.dumps(metrics, indent=2))

model.save('/content/models/disease_model.keras')
Path('/content/models/class_names.json').write_text(json.dumps(class_names, indent=2))
print('Trained model and class names successfully saved!')

### 📦 Step 4: Zip & Download Trained Artifacts

In [ ]:
# Create zip archive containing model and report figures
!mkdir -p /content/biovision_artifacts/models /content/biovision_artifacts/reports
!cp /content/models/* /content/biovision_artifacts/models/
!cp /content/reports/* /content/biovision_artifacts/reports/
!zip -r biovision_trained_artifacts.zip /content/biovision_artifacts

from google.colab import files
files.download('biovision_trained_artifacts.zip')
print('Download initiated! Extract the contents into your project folder.')